# Score Following with Hidden Markov Models

In this notebook we will explore online alignment with Hidden Markov Models.

Alignment problems can be organized according to how they process information as:

* **Offline**: Alignment of two *recordings/documents* (i.e., audio recordings, MIDI performances, MusicXML scores, etc.). These recordings/documents can be in any of the modalities described above, the important thing being that the music is occurring in real-time.

* **Online**: Alignment of a live (i.e., real time) performance to the music encoded in a target document (e.g., a pre-annotated audio recording, a symbolic score, etc.). The problem of real time online alignment is known in the MIR literature a **score following**, and can be useful in live interactive settings, such as automatic accompaniment systems

In this tutorial we are going to focus on the case of **online alignment**.

In [ ]:
import os
import warnings
from typing import Any, Callable, Dict, Generator, List, Optional, Tuple, Union

import numpy as np

from hiddenmarkov import (
    ConstantTransitionModel,
    ObservationModel,
    TransitionModel,
)
from numpy.typing import NDArray
from scipy.signal import convolve
from scipy.stats import gumbel_l, norm

import partitura as pt
import matplotlib.pyplot as plt

from hmm_utils import (
    BaseHMM,
    TempoModel,
    KalmanTempoModel,
    interleave_with_constant,
    compute_ioi_matrix,
    compute_chord_matrix,
    transfer_positions,
)

warnings.filterwarnings("ignore")
# Alias for typing arrays
NDArrayFloat = NDArray[np.float32]
NDArrayInt = NDArray[np.int32]

## Observation Model

Before we define an observation model, we should think first:

* What should be the hidden states of the model?
* What are our observations?
* How do we compute $p(\mathbf{x}_n \mid \mathbf{z}_k)$? (i.e., the likelihood of observing $\mathbf{x}_n$ given hidden state $\mathbf{z}_k$)


Let's start with the first question

### Defining states

The obvious choice is for the hidden states to represent score positions.


In [ ]:
def compute_discrete_pitch_profiles(
    chord_pitches: NDArrayFloat,
    profile: NDArrayFloat = np.array([0.02, 0.02, 1, 0.02, 0.02]),
    eps: float = 0.01,
    piano_range: bool = False,
    normalize: bool = True,
    inserted_states: bool = True,
) -> NDArrayFloat:
    """
    Pre-compute the pitch profiles used in calculating the pitch
    observation probabilities.

    Parameters
    ----------
    chord_pitches : numpy array
        A 2D array of size (n_chords, 128) with the pitch of each chord.

    profile : numpy array
        The probability "gain" of how probable are the closest pitches to
        the one in question.

    eps : float
        The epsilon value to be added to each pre-computed pitch profile.

    piano_range : boolean
        Indicates whether the possible MIDI pitches are to be restricted
        within the range of a piano.

    normalize : boolean
        Indicates whether the pitch profiles are to be normalized.

    Returns
    -------
    pitch_profiles : numpy array
        The pre-computed pitch profiles.
    """

    chord_matrix = compute_chord_matrix(chord_pitches=chord_pitches)

    pitch_profiles = convolve(chord_matrix, profile[None, :], mode="same")

    if inserted_states:
        pitch_profiles = interleave_with_constant(array=pitch_profiles)
        pitch_profiles = pitch_profiles[:-1]
    # Add extra value
    pitch_profiles += eps

    # Check whether to trim and normalize:
    if piano_range:
        pitch_profiles = pitch_profiles[:, 21:109]
    if normalize:
        pitch_profiles /= pitch_profiles.sum(1, keepdims=True)

    return pitch_profiles



Let's look at an example

In [ ]:
score_fn = os.path.join(".", "example_data", "mozart_k265_var1.musicxml")

score = pt.load_musicxml(score_fn)
snote_array = score.note_array()

In [ ]:
unique_sonsets = np.unique(snote_array["onset_beat"])
unique_sonset_idxs = [
    np.where(snote_array["onset_beat"] == ui)[0] for ui in unique_sonsets
]
chord_pitches = [snote_array["pitch"][uix] for uix in unique_sonset_idxs]

piano_range = True
inserted_states = False
pitch_profiles = compute_discrete_pitch_profiles(
    chord_pitches=chord_pitches,
    piano_range=piano_range,
    inserted_states=inserted_states,
)

In [ ]:
plt.imshow(
    pitch_profiles.T,
    aspect="auto",
    origin="lower",
    cmap="gray",
    interpolation="nearest",
)
plt.title("Pitch Profiles")
plt.ylabel("Piano Key")
plt.xlabel("Score Onset Index")
plt.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def plot_pitch_profile(i):
    plt.figure(figsize=(8, 4))
    plt.plot(pitch_profiles[i])
    plt.xlabel("Piano key")
    plt.ylabel("Pitch likelihood")
    plt.title(f"Score Position {unique_sonsets[i]:.2f} beats")
    plt.show()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(pitch_profiles) - 1,
    step=1,
    description="Position",
    continuous_update=False  # avoids lag
)

widgets.interact(plot_pitch_profile, i=slider)

We can use these profiles 

In [ ]:
def compute_bernoulli_pitch_probabilities(
    pitch_obs: NDArrayFloat,
    pitch_profiles: NDArrayFloat,
) -> NDArrayFloat:
    """
    Compute pitch observation probabilities
    """

    # Compute Bernoulli probability:
    pitch_prob = (pitch_profiles**pitch_obs) * ((1 - pitch_profiles) ** (1 - pitch_obs))

    obs_prob = np.prod(pitch_prob, 1)

    return obs_prob


In [ ]:
class BernoulliPitchObservationModel(ObservationModel):
    """
    Computes the probabilities that an observation was emitted, i.e. the
    likelihood of observing performed notes at the current moment/state.

    Parameters
    ----------
    pitch_profiles : NDArrayFloat
        Pre-computed pitch profiles, for each separate possible pitch
        in the MIDI range. Used in calculating the pitch observation
        probabilities.
    """

    def __init__(self, pitch_profiles: NDArrayFloat):
        """
        The initialization method.

        Parameters
        ----------
        pitch_profiles : NDArrayFloat
            he pre-computed pitch profiles, for each separate possible pitch
            in the MIDI range. Used in calculating the pitch observation
            probabilities.
        """
        super().__init__(use_log_probabilities=False)
        # Store the parameters of the object:
        self.pitch_profiles = pitch_profiles

    def __call__(self, observation: NDArrayFloat) -> NDArrayFloat:
        return compute_bernoulli_pitch_probabilities(
            pitch_obs=observation,
            pitch_profiles=self.pitch_profiles,
        )

Let's visualize this by looking at how the pitch probabilities would change for each note in the piano:

In [ ]:
from partitura.utils.music import (
    midi_pitch_to_pitch_spelling,
    pitch_spelling_to_note_name,
)

observation_model = BernoulliPitchObservationModel(
    pitch_profiles=pitch_profiles,
)

# Piano key labels (A0 = MIDI 21)
x_labels = [
    pitch_spelling_to_note_name(*midi_pitch_to_pitch_spelling(i + 21))
    for i in range(88)
]

def plot_obs_prob(key):
    pitch_obs = np.zeros(88)
    pitch_obs[key] = 1

    obs_prob = observation_model(pitch_obs)
    n = len(obs_prob)

    plt.figure(figsize=(10, 4))
    plt.plot(obs_prob)

    plt.xlabel("Frame index")
    plt.ylabel("Observation probability")
    plt.title(
        f"Active piano key: "
        f"{pitch_spelling_to_note_name(*midi_pitch_to_pitch_spelling(key + 21))}"
    )
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=87,
    step=1,
    description="Key",
    continuous_update=False,
)

widgets.interact(plot_obs_prob, key=slider)

**Question**: What is this plot telling us?

## Transition Model

For a regular performance, we want something that satisfies the following conditions

1. It gives a high probability to moving to the next state (i.e., penalize large jumps or going backwards)
2. It gives a non-zero probability to staying in the same state
3. Gives a low probability to the other states.

There are many possibilities. Here we are going to use one using a Gumbel distribution.

In [ ]:
def gumbel_transition_matrix(
    n_states: int,
    mp_trans_state: int = 1,
    scale: float = 0.5,
    inserted_states: bool = False,
) -> NDArrayFloat:
    """
    Compute a transiton matrix, where each row follows a normalized Gumbel
    distribution.

    Parameters
    ----------
    n_states : int
        The number of states in the Hidden Markov Model (HMM), which is required
        for the size of the matrix.

    mp_trans_state : int
        Which state should have the largest probability to be transitioned into
        from the current state the model is in.
        Default = 1, which means that the model would prioritize transitioning
        into the state that is next in line, e.g. from State 3 to State 4.

    scale : float
        The scale parameter of the distribution.
        Default = 0.5

    inserted_states : boolean
        Indicates whether the HMM includes inserted states (intermediary states
        between chords for errors and insertions in the score following).
        Default = True

    Returns
    -------
    transition_matrix : numpy array
        The computed transition matrix for the HMM.
    """
    # Initialize transition matrix:
    transition_matrix = np.zeros((n_states, n_states), dtype="f8")

    # Compute transition matrix:
    for i in range(n_states):
        if inserted_states:
            if np.mod(i, 2) == 0:
                transition_matrix[i] = gumbel_l.pdf(
                    np.arange(n_states), loc=i + mp_trans_state * 2, scale=scale
                )
            else:
                transition_matrix[i] = gumbel_l.pdf(
                    np.arange(n_states), loc=i + mp_trans_state * 2 - 1, scale=scale
                )
        else:
            transition_matrix[i] = gumbel_l.pdf(
                np.arange(n_states), loc=i + mp_trans_state * 2 - 1, scale=scale
            )

    # Normalize transition matrix (so that it is a proper stochastic matrix):
    transition_matrix /= transition_matrix.sum(1, keepdims=True)

    # Return the computed transition matrix:
    return transition_matrix


def gumbel_init_dist(
    n_states: int,
    loc: int = 0,
    scale: float = 10,
) -> NDArrayFloat:
    """
    Compute the initial probabilites for all states in the Hidden Markov Model
    (HMM), which follow a Gumbel distribution.

    Parameters
    ----------
    n_states : int
        The number of states in the Hidden Markov Model (HMM), which is required
        for the size of the initial probabilites vector.

    Returns
    -------
    init_probs : numpy array
        The computed initial probabilities in the form of a vector.
    """

    prob_scale: float = scale if scale < n_states else n_states / 10

    init_probs: np.ndarray = gumbel_l.pdf(
        np.arange(n_states),
        loc=loc,
        scale=prob_scale,
    )

    return init_probs

Let's see how this probability looks like:

* What happens if you change the parameters `loc` and `scale`?

In [ ]:
n_states = 10
loc = 1
scale = 0.5
distribution = gumbel_l.pdf(
    np.arange(n_states),
    loc=loc,
    scale=scale,
)

plt.plot(distribution)
plt.xlabel("Values")
plt.ylabel("pdf")

In [ ]:
transition_matrix = gumbel_transition_matrix(
    n_states=10,
    mp_trans_state=1,
)

plt.imshow(
    transition_matrix,
    aspect="equal",
    cmap="magma",
)
plt.xlabel("Next state")
plt.ylabel("Starting state")

### Putting it all together

Let's put together an HMM based on this observation and transition models.

In [ ]:
class PitchHMM(BaseHMM):
    """
    Implements the behavior of a HiddenMarkovModel, specifically designed for
    the task of score following.

    Parameters
    ----------
    _transition_matrix : numpy.ndarray
        Matrix for computations of state transitions within the HMM.

    _observation_model : ObservationModel
        Object responsible for computing the observation probabilities for each
        state of the HMM.

    initial_distribution : numpy array
        The initial distribution of the model. If not given, it is assumed to
        be uniform.

    forward_variable : numpy array
        The current (latest) value of the forward variable.

    _variation_coeff : float
        The normalized coefficient of variation of the current (latest) forward
        variable. Used to determine the confidence of the prediction of the HMM.

    current_state : int
        The index of the current state of the HMM.
    """

    def __init__(
        self,
        reference_features: np.ndarray,  # snote_array
        transition_model: Optional[TransitionModel] = None,
        observation_model: Optional[ObservationModel] = None,
        transition_matrix: Optional[NDArrayFloat] = None,
        pitch_obs_prob_func: Optional[Callable[..., NDArrayFloat]] = None,
        pitch_prob_args: Optional[Dict[str, Any]] = None,
        initial_probabilities: Optional[np.ndarray] = None,
        has_insertions: bool = True,
        piano_range: bool = True,
    ) -> None:
        """
        Initialize the object.

        Parameters
        ----------
        transition_matrix : numpy array
            The Tranistion probability matrix of HMM.

        pitch_profiles : numpy array
            The pre-computed pitch profiles, for each separate possible pitch
            in the MIDI range. Used in calculating the pitch observation
            probabilities.

        ioi_matrix : numpy array
            The pre-computed score IOI values in beats, from each unique state
            to all other states, stored in a matrix.

        ioi_precision : float
            The precision parameter for computing the IOI observation
            probability.

        score_onsets : numpy array

        initial_distribution : numpy array
            The initial distribution of the model. If not given, it is asumed to
            be uniform.
            Default = None.
        """

        self.reference_features = reference_features
        (
            observation_model,
            transition_matrix,
            initial_probabilities,
            unique_onsets,
        ) = self._build_hmm_modules(
            inserted_states=has_insertions,
            piano_range=piano_range,
        )

        if transition_model is not None and transition_matrix is not None:
            warnings.warn(
                "Both `transition_model` and `transition_matrix` were "
                "provided. Only `transition_model` will be used."
            )
        obs_model_params_given = [
            pitch_obs_prob_func is not None,
            pitch_prob_args is not None,
        ]
        if observation_model is not None and any(obs_model_params_given):
            warnings.warn(
                "`observation_model` and params were provided. "
                "Only `observation_model` will be used."
            )

        if observation_model is None and not all(obs_model_params_given):
            missing_params = [
                pn
                for pn, given in zip(
                    [
                        "pitch_obs_prob_func",
                        "pitch_prob_args",
                    ],
                    obs_model_params_given,
                )
                if not given
            ]
            raise ValueError(missing_params)

        if transition_model is None:
            transition_model = ConstantTransitionModel(
                transition_probabilities=transition_matrix,
                init_probabilities=initial_probabilities,
            )


        self.perf_onset = None

        BaseHMM.__init__(
            self,
            observation_model=observation_model,
            transition_model=transition_model,
            state_space=unique_onsets,
            has_insertions=has_insertions,
        )

    def __call__(self, input, *args, **kwargs):
        frame_index = args[0] if args else None
        pitch_obs, ioi_obs = input

        if self.perf_onset is None:
            self.perf_onset = 0
        else:
            self.perf_onset += ioi_obs
        current_state = self.forward_algorithm_step(
            observation=pitch_obs,
            log_probabilities=False,
        )
        self._warping_path.append((current_state, self.input_index))
        self.input_index = self.input_index + 1 if frame_index is None else frame_index

        self.current_state = current_state

        return self.current_state

    @property
    def current_state(self):
        return self.observation_model.current_state

    @current_state.setter
    def current_state(self, state):
        self.observation_model.current_state = state

    def _build_hmm_modules(
        self,
        piano_range: bool = True,
        inserted_states: bool = True,
    ):
        snote_array = self.reference_features
        unique_sonsets = np.unique(snote_array["onset_beat"])
        unique_sonset_idxs = [
            np.where(snote_array["onset_beat"] == ui)[0] for ui in unique_sonsets
        ]
        chord_pitches = [snote_array["pitch"][uix] for uix in unique_sonset_idxs]
        pitch_profiles = compute_discrete_pitch_profiles(
            chord_pitches=chord_pitches,
            piano_range=piano_range,
            inserted_states=inserted_states,
        )

        # observation model
        observation_model = BernoulliPitchObservationModel(
            pitch_profiles=pitch_profiles,
        )

        if inserted_states:
            unique_onsets_s = np.insert(
                unique_sonsets,
                np.arange(1, len(unique_sonsets)),
                (unique_sonsets[:-1] + 0.5 * np.diff(unique_sonsets)),
            )
        else:
            unique_onsets_s = unique_sonsets

        transition_matrix = gumbel_transition_matrix(
            n_states=len(unique_onsets_s),
            inserted_states=True,
        )
        initial_probabilities = gumbel_init_dist(
            n_states=len(unique_onsets_s),
        )

        return (
            observation_model,
            transition_matrix,
            initial_probabilities,
            unique_onsets_s,
        )

Let's look at an example

In [ ]:
from features.midi import compute_features_from_symbolic

match_fn = os.path.join(".", "example_data", "mozart_k265_var1.match")
score_fn = os.path.join(".", "example_data", "mozart_k265_var1.musicxml")

perf, alignment = pt.load_match(match_fn)
score = pt.load_musicxml(score_fn)

The following function "simulates" real time input, and returns a list with the inputs that we would expect in real time, so we just need to iterate through them. The `polling_period` argument specifies how often do we "look" into the input stream. It can be seen as some sort of 

In [ ]:
polling_period = 0.01
perf_features = compute_features_from_symbolic(
    ref_info=perf,
    processor_name="pitch_ioi",
    processor_kwargs={
        "piano_range": True,
    },
    polling_period=0.01,
)

Let's have a look at the features.

In [ ]:
# indices of valid (non-None) features
valid_idx = [i for i, feat in enumerate(perf_features) if feat is not None]

# precompute accumulated onsets
perf_onsets = []
perf_onset = None

for i in valid_idx:
    _, ioi_obs = perf_features[i]

    if perf_onset is None:
        perf_onset = ioi_obs
    else:
        perf_onset += ioi_obs

    perf_onsets.append(perf_onset)

def plot_feat(j):
    i = valid_idx[j]
    pitch_obs, ioi_obs = perf_features[i]
    perf_onset = perf_onsets[j]

    obs_prob = observation_model(pitch_obs)

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)

    # ---- top: observation probability
    axes[0].plot(obs_prob)
    axes[0].set_ylabel("Observation probability")
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"perf_features[{i}]")

    # ---- bottom: accumulated onset
    axes[1].scatter(j, perf_onset)
    axes[1].plot(perf_onsets[: j + 1], marker="o")
    axes[1].set_xlabel("Feature index")
    axes[1].set_ylabel("Performed onset (s)")
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(valid_idx) - 1,
    step=1,
    description="Feature",
    continuous_update=False,
)

widgets.interact(plot_feat, j=slider)

Let's initialize our `PitchHMM` object and evaluate it.

In [ ]:
pitchhmm = PitchHMM(
    reference_features=score.note_array(),
    has_insertions=False,
    piano_range=True,
)

current_position_ph = pitchhmm.current_state
# A list to store the alignment results
alignment_ph = []

# for each observation
for i, feat in enumerate(perf_features):
    if feat is not None:
        current_position_ph = pitchhmm(feat)
    alignment_ph.append((i, i * polling_period, pitchhmm.state_space[current_position_ph]))

alignment_ph = np.array(alignment_ph)

In [ ]:
def interactive_alignment(alignment):
    """
    Interactive slider plot for alignment array.

    Parameters
    ----------
    alignment_ph : np.ndarray
        Shape (N, 3): columns [i, time, position]
    """
    alignment_ph = np.asarray(alignment)
    idxs = alignment[:, 0].astype(int)
    times = alignment[:, 1].astype(float)
    positions = alignment[:, 2].astype(float)

    def plot_step(step):
        plt.figure(figsize=(10, 4))
        plt.plot(times[: step + 1], positions[: step + 1], lw=2)
        plt.plot([times[step]], [positions[step]], marker="o")

        plt.xlabel("Time (s)")
        plt.ylabel("Aligned position")
        plt.title(f"Step {step} (i={idxs[step]}, t={times[step]:.3f}s)")
        plt.tight_layout()
        plt.show()

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(times) - 1,
        step=1,
        description="Step",
        continuous_update=False,
    )

    return widgets.interact(plot_step, step=slider)

interactive_alignment(alignment_ph)





## Expanding the Model: PitchIOI HMM

The previous HMM only considered pitch information. An extension of this model would use both time and pitch information.

We are going to use the performed IOI (inter-onset-interval) for this. We can simply model it as two independent events

$$p(\text{pitch}, \text{IOI} \mid \text{score pos}) = p(\text{pitch} \mid \text{score pos}) \cdot p(\text{IOI} \mid \text{score pos})$$

In this case, we can keep our same Bernoulli probabilities for the pitch observation, and add a simple Gaussian Model for the IOI.

In [ ]:
def compute_gaussian_ioi_observation_probability(
    ioi_obs: float,
    ioi_score: NDArrayFloat,
    tempo_est: float,
    ioi_precision: float,
    norm_term: float,
) -> NDArrayFloat:
    """
    Compute the IOI observation probability as a zero mean
    Gaussian

    Parameters
    ----------
    ioi_obs : numpy array
        All the observed IOI.

    current_state : int
        The current state of the Score HMM.

    tempo_est : float
        The tempo estimation.

    Returns
    -------
    obs_prob : numpy array
        The computed IOI observation probabilities for each state.
    """
    # Compute the expected argument:
    exp_arg = -0.5 * ((tempo_est * ioi_score - ioi_obs) ** 2) * ioi_precision

    obs_prob = norm_term * np.exp(exp_arg)
    return obs_prob

In [ ]:
class PitchIOIObservationModel(ObservationModel):
    def __init__(
        self,
        pitch_obs_prob_func: Callable[..., NDArrayFloat],
        ioi_obs_prob_func: Callable[..., NDArrayFloat],
        ioi_matrix: NDArrayFloat,
        pitch_prob_args: Optional[Dict[str, Any]] = None,
        ioi_prob_args: Optional[Dict[str, Any]] = None,
    ) -> None:
        super().__init__(use_log_probabilities=False)

        self.pitch_obs_prob_func = pitch_obs_prob_func
        self.ioi_obs_prob_func = ioi_obs_prob_func
        self.pitch_prob_args = pitch_prob_args
        self.ioi_prob_args = ioi_prob_args
        self.ioi_matrix = ioi_matrix
        self.current_state = None

    def __call__(self, observation: Any, *args, **kwargs) -> NDArrayFloat:
        pitch_obs, ioi_obs, tempo_est = observation
        ioi_idx = self.current_state if self.current_state is not None else 0

        ioi_score = self.ioi_matrix[ioi_idx]
        obs_prob = self.pitch_obs_prob_func(
            pitch_obs=pitch_obs,
            **self.pitch_prob_args,
        ) * self.ioi_obs_prob_func(
            ioi_obs=ioi_obs,
            ioi_score=ioi_score,
            tempo_est=tempo_est,
            **self.ioi_prob_args,
        )
        return obs_prob


class BernoulliGaussianPitchIOIObservationModel(PitchIOIObservationModel):
    def __init__(self, pitch_profiles, ioi_matrix, ioi_precision):
        """
        The initialization method.

        Parameters
        ----------
        pitch_profiles : numpy array
            he pre-computed pitch profiles, for each separate possible pitch
            in the MIDI range. Used in calculating the pitch observation
            probabilities.

        ioi_matrix : numpy array
            The pre-computed score IOI values in beats, from each unique state
            to all other states, stored in a matrix.

        ioi_precision : float
            The precision parameter for computing the IOI observation
            probability.
        """

        pitch_prob_args = dict(
            pitch_profiles=pitch_profiles,
        )
        ioi_prob_args = dict(
            ioi_precision=ioi_precision,
            norm_term=np.sqrt(0.5 * ioi_precision / np.pi),
        )
        PitchIOIObservationModel.__init__(
            self,
            pitch_obs_prob_func=compute_bernoulli_pitch_probabilities,
            ioi_obs_prob_func=compute_gaussian_ioi_observation_probability,
            ioi_matrix=ioi_matrix,
            pitch_prob_args=pitch_prob_args,
            ioi_prob_args=ioi_prob_args,
        )


Le'ts visualize this!

In [ ]:
unique_sonsets = np.unique(snote_array["onset_beat"])
unique_sonset_idxs = [
    np.where(snote_array["onset_beat"] == ui)[0] for ui in unique_sonsets
]
chord_pitches = [snote_array["pitch"][uix] for uix in unique_sonset_idxs]
pitch_profiles = compute_discrete_pitch_profiles(
    chord_pitches=chord_pitches,
    piano_range=piano_range,
    inserted_states=inserted_states,
)
ioi_matrix = compute_ioi_matrix(
    unique_onsets=unique_sonsets,
    inserted_states=inserted_states,
)

# observation model
observation_model = BernoulliGaussianPitchIOIObservationModel(
    pitch_profiles=pitch_profiles,
    ioi_matrix=ioi_matrix,
    ioi_precision=1,
)

In [ ]:
# indices of valid (non-None) features
valid_idx = [i for i, feat in enumerate(perf_features) if feat is not None]

# precompute accumulated onsets
perf_onsets = []
perf_onset = None

for i in valid_idx:
    _, ioi_obs = perf_features[i]

    if perf_onset is None:
        perf_onset = ioi_obs
    else:
        perf_onset += ioi_obs

    perf_onsets.append(perf_onset)

def plot_feat(j):
    i = valid_idx[j]
    pitch_obs, ioi_obs = perf_features[i]
    perf_onset = perf_onsets[j]

    obs_prob = observation_model((pitch_obs, ioi_obs, ioi_obs /4))

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)

    # ---- top: observation probability
    axes[0].plot(obs_prob)
    axes[0].set_ylabel("Observation probability")
    axes[0].set_ylim(0, 0.3)
    axes[0].set_title(f"perf_features[{i}]")

    # ---- bottom: accumulated onset
    axes[1].scatter(j, perf_onset)
    axes[1].plot(perf_onsets[: j + 1], marker="o")
    axes[1].set_xlabel("Feature index")
    axes[1].set_ylabel("Performed onset (s)")
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(valid_idx) - 1,
    step=1,
    description="Feature",
    continuous_update=False,
)

widgets.interact(plot_feat, j=slider)

Let's put everything together.

In [ ]:
class PitchIOIHMM(BaseHMM):
    """
    Implements the behavior of a HiddenMarkovModel, specifically designed for
    the task of score following.

    Parameters
    ----------
    _transition_matrix : numpy.ndarray
        Matrix for computations of state transitions within the HMM.

    _observation_model : ObservationModel
        Object responsible for computing the observation probabilities for each
        state of the HMM.

    initial_distribution : numpy array
        The initial distribution of the model. If not given, it is assumed to
        be uniform.

    forward_variable : numpy array
        The current (latest) value of the forward variable.

    _variation_coeff : float
        The normalized coefficient of variation of the current (latest) forward
        variable. Used to determine the confidence of the prediction of the HMM.

    current_state : int
        The index of the current state of the HMM.
    """

    def __init__(
        self,
        reference_features: np.ndarray,  # snote_array
        tempo_model: TempoModel = None,
        transition_model: Optional[TransitionModel] = None,
        observation_model: Optional[PitchIOIObservationModel] = None,
        transition_matrix: Optional[NDArrayFloat] = None,
        pitch_obs_prob_func: Optional[Callable[..., NDArrayFloat]] = None,
        ioi_obs_prob_func: Optional[Callable[..., NDArrayFloat]] = None,
        ioi_matrix: Optional[NDArrayFloat] = None,
        pitch_prob_args: Optional[Dict[str, Any]] = None,
        ioi_prob_args: Optional[Dict[str, Any]] = None,
        initial_probabilities: Optional[np.ndarray] = None,
        has_insertions: bool = True,
        piano_range: bool = True,
    ) -> None:
        """
        Initialize the object.

        Parameters
        ----------
        transition_matrix : numpy array
            The Tranistion probability matrix of HMM.

        pitch_profiles : numpy array
            The pre-computed pitch profiles, for each separate possible pitch
            in the MIDI range. Used in calculating the pitch observation
            probabilities.

        ioi_matrix : numpy array
            The pre-computed score IOI values in beats, from each unique state
            to all other states, stored in a matrix.

        ioi_precision : float
            The precision parameter for computing the IOI observation
            probability.

        score_onsets : numpy array
            TODO

        initial_distribution : numpy array
            The initial distribution of the model. If not given, it is asumed to
            be uniform.
            Default = None.
        """
        self.reference_features = reference_features

        (
            observation_model,
            transition_matrix,
            initial_probabilities,
            tempo_model,
            unique_onsets,
        ) = self._build_hmm_modules(
            inserted_states=has_insertions,
            piano_range=piano_range,
        )

        if transition_model is not None and transition_matrix is not None:
            warnings.warn(
                "Both `transition_model` and `transition_matrix` were "
                "provided. Only `transition_model` will be used."
            )
        obs_model_params_given = [
            pitch_obs_prob_func is not None,
            ioi_obs_prob_func is not None,
            ioi_matrix is not None,
            pitch_prob_args is not None,
            ioi_prob_args is not None,
        ]
        if observation_model is not None and any(obs_model_params_given):
            warnings.warn(
                "`observation_model` and params were provided. "
                "Only `observation_model` will be used."
            )

        if observation_model is None and not all(obs_model_params_given):
            missing_params = [
                pn
                for pn, given in zip(
                    [
                        "pitch_obs_prob_func",
                        "ioi_obs_prob_func",
                        "ioi_matrix",
                        "pitch_prob_args",
                        "ioi_prob_args",
                    ],
                    obs_model_params_given,
                )
                if not given
            ]
            raise ValueError(missing_params)

        if transition_model is None:
            transition_model = ConstantTransitionModel(
                transition_probabilities=transition_matrix,
                init_probabilities=initial_probabilities,
            )

        if observation_model is None:
            observation_model = PitchIOIObservationModel(
                pitch_obs_prob_func=pitch_obs_prob_func,
                ioi_obs_prob_func=ioi_obs_prob_func,
                ioi_matrix=ioi_matrix,
                pitch_prob_args=pitch_prob_args,
                ioi_prob_args=ioi_prob_args,
            )

        self.perf_onset = None

        BaseHMM.__init__(
            self,
            observation_model=observation_model,
            transition_model=transition_model,
            state_space=unique_onsets,
            tempo_model=tempo_model,
            has_insertions=has_insertions,
        )

    def __call__(self, input, *args, **kwargs):
        frame_index = args[0] if args else None
        pitch_obs, ioi_obs = input

        if self.perf_onset is None:
            self.perf_onset = 0
        else:
            self.perf_onset += ioi_obs
        current_state = self.forward_algorithm_step(
            observation=(
                pitch_obs,
                ioi_obs,
                self.tempo_model.beat_period,
            ),
            log_probabilities=False,
        )
        self._warping_path.append((current_state, self.input_index))
        self.input_index = self.input_index + 1 if frame_index is None else frame_index

        if self.current_state is None:
            self.current_state = current_state

        if (
            current_state > self.current_state
        ):  # TODO: check if it works for audio (current state moves?) -> transition matrix
            if self.has_insertions and current_state % 2 == 0:
                current_so = self.state_space[current_state]
                # prev_so = self.state_space[self.current_state]

                self.tempo_model.update_beat_period(
                    performed_onset=self.perf_onset,
                    score_onset=current_so,
                )

            elif not self.has_insertions:
                current_so = self.state_space[current_state]
                self.tempo_model.update_beat_period(
                    performed_onset=self.perf_onset,
                    score_onset=current_so,
                )

        self.current_state = current_state

        return self.current_state

    @property
    def current_state(self):
        return self.observation_model.current_state

    @current_state.setter
    def current_state(self, state):
        self.observation_model.current_state = state

    def _build_hmm_modules(
        self,
        piano_range: bool = True,
        inserted_states: bool = True,
    ):
        snote_array = self.reference_features
        unique_sonsets = np.unique(snote_array["onset_beat"])
        unique_sonset_idxs = [
            np.where(snote_array["onset_beat"] == ui)[0] for ui in unique_sonsets
        ]
        chord_pitches = [snote_array["pitch"][uix] for uix in unique_sonset_idxs]
        pitch_profiles = compute_discrete_pitch_profiles(
            chord_pitches=chord_pitches,
            piano_range=piano_range,
            inserted_states=inserted_states,
        )
        ioi_matrix = compute_ioi_matrix(
            unique_onsets=unique_sonsets,
            inserted_states=inserted_states,
        )

        # observation model
        observation_model = BernoulliGaussianPitchIOIObservationModel(
            pitch_profiles=pitch_profiles,
            ioi_matrix=ioi_matrix,
            ioi_precision=1,
        )

        if inserted_states:
            unique_onsets_s = np.insert(
                unique_sonsets,
                np.arange(1, len(unique_sonsets)),
                (unique_sonsets[:-1] + 0.5 * np.diff(unique_sonsets)),
            )
        else:
            unique_onsets_s = unique_sonsets

        # tempo model
        tempo_model = KalmanTempoModel(
            init_score_onset=unique_sonsets.min(),
            init_beat_period=60 / 100,
        )
        transition_matrix = gumbel_transition_matrix(
            n_states=len(ioi_matrix[0]),
            inserted_states=True,
        )
        initial_probabilities = gumbel_init_dist(
            n_states=len(ioi_matrix[0]),
        )

        return (
            observation_model,
            transition_matrix,
            initial_probabilities,
            tempo_model,
            unique_onsets_s,
        )

In [ ]:
pitchioihmm = PitchIOIHMM(
    reference_features=score.note_array(),
)

current_position_pih = pitchioihmm.current_state
alignment_pih = []
for i, feat in enumerate(perf_features):

    if feat is not None:
        current_position_pih = pitchioihmm(feat)
    alignment_pih.append((i, i * polling_period, pitchioihmm.state_space[current_position_pih]))

alignment_pih = np.array(alignment_pih)

In [ ]:
interactive_alignment(alignment_pih)

In [ ]:
plt.plot(
    alignment_ph[:, 1],
    alignment_ph[:, 2],
    color="firebrick",
    label="PitchHMM",
    linewidth=2,
)
plt.plot(
    alignment_pih[:, 1],
    alignment_pih[:, 2],
    color="navy",
    label="PitchIOIHMM",
    linewidth=2,
)
plt.xlabel("Time (s)")
plt.ylabel("Position (beats)")
plt.legend()
plt.show()